In [24]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic
import ast
import re


load_dotenv()

client = Anthropic()
model = "claude-haiku-5-5"

In [25]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, schema=None):
    params = {
        "model": model,
        "max_tokens": 16000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    # Structured outputs replace the old "prefill + stop sequence" trick:
    # the API guarantees the response text is JSON matching this schema.
    if schema:
        params["output_config"] = {"format": {"type": "json_schema", "schema": schema}}

    message = client.messages.create(**params)
    # The model may think first, so pick the text blocks instead of content[0]
    return "".join(block.text for block in message.content if block.type == "text")

In [ ]:
# Function to generate a new dataset
import json

# Shape we want back. The top level must be an object, so the array of
# tasks lives under a "tasks" key.
dataset_schema = {
    "type": "object",
    "properties": {
        "tasks": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "task": {"type": "string"},
                    "format": {"type": "string", "enum": ["json", "python", "regex"]},
                    "solution_criteria": {"type": "string"},
                    "restriction": {"type": "string"}
                },
                "required": ["task", "format", "solution_criteria", "restriction"],
                "additionalProperties": False,
            },
        }
    },
    "required": ["tasks"],
    "additionalProperties": False,
}


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
        "solution_ßcriteria": "Description of what the solution should accomplish"
        "restriction": "Description of any restrictions or limitations on the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    text = chat(messages, schema=dataset_schema)
    return json.loads(text)["tasks"]

In [38]:
# Generate the dataset and write it to 'dataset.json'
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [ ]:
# Shape of the grade we want back from the grader model
grade_schema = {
    "type": "object",
    "properties": {
        "strengths": {"type": "array", "items": {"type": "string"}},
        "weaknesses": {"type": "array", "items": {"type": "string"}},
        "reasoning": {"type": "string"},
        "score": {"type": "integer"},
    },
    "required": ["strengths", "weaknesses", "reasoning", "score"],
    "additionalProperties": False,
}


# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria the solution must meet:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Restrictions the solution must respect:
<restrictions>
{test_case["restriction"]}
</restrictions>

Grade the solution against the criteria and the restrictions above, not against your own idea of a good answer.
A solution that misses part of the criteria or breaks any restriction must lose points, even if it otherwise works.

In your evaluation:
- "strengths": 1-3 key strengths
- "weaknesses": 1-3 key areas for improvement
- "reasoning": a concise explanation of how the solution does or does not meet the criteria and restrictions
- "score": an integer from 1 to 10

Keep your response concise and direct.
    """

    messages = []
    add_user_message(messages, eval_prompt)
    eval_text = chat(messages, schema=grade_schema)
    return json.loads(eval_text)

In [29]:
# Passes a test case into Claude
solution_schema = {
    "type": "object",
    "properties": {
        "code": {"type": "string"},
    },
    "required": ["code"],
    "additionalProperties": False,
}

def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond with only Python, JSON, or a plain Regex
* Do not add any commnets or commentary or explanation

"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages, schema = solution_schema)
    return json.loads(output)["code"]

In [30]:
def validate_json(text):
    try:
        json.loads(text)
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(response, test_case):\

    format_type = test_case["format"]
    if format_type == "json":
        return validate_json(response)
    elif format_type == "python":
        return validate_python(response)
    elif format_type == "regex":
        return validate_regex(response)
    else:
        raise ValueError(f"Unknown format type: {format_type}") 

In [31]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)
    score = (model_score + syntax_score) / 2


    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [32]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [33]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 9.5


In [34]:
print(json.dumps(results, indent=2))

[
  {
    "output": "def parse_s3_keys(keys):\n    result = {}\n    for key in keys:\n        prefix = key.split('/', 1)[0] if '/' in key else ''\n        result[prefix] = result.get(prefix, 0) + 1\n    return result",
    "test_case": {
      "task": "Write a single Python function named parse_s3_keys that takes a list of S3 object key strings and returns a dictionary mapping each top-level prefix (the part before the first slash, or the empty string if there is none) to the count of keys under it. Solution criteria: correct grouping and counting for keys like logs/2024/a.txt and root.txt. Restriction: use only the standard library.",
      "format": "python"
    },
    "score": 9.5,
    "reasoning": "The function meets the criteria. For logs/2024/a.txt it yields 'logs', and for root.txt it yields ''. Leading-slash keys like '/a' map to '' as well, which is consistent with splitting on the first slash. Grouping and counting are correct, the code is concise, and it relies only on the s